# 03 - Spatial segmentation

Four segmentations of the same foreground pixels, then an honest measure of
how much they agree.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [1]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
# Show paths relative to the project root so notebooks stay portable.
print("Project  :", ROOT.name)
print("Raw data :", config.raw_dir.relative_to(ROOT) if ROOT in config.raw_dir.parents else config.raw_dir)
print("Outputs  :", config.processed_dir.relative_to(ROOT) if ROOT in config.processed_dir.parents else config.processed_dir)

Project  : Fossil Fly
Raw data : data\raw
Outputs  : data\processed


In [2]:
import numpy as np

from src.features import decomposition as decomp
from src.segmentation import cluster

artefacts = decomp.load_decomposition(config.processed_dir)
fg_mask = artefacts["fg_mask"].astype(bool)
fg_idx = np.where(fg_mask)[0]

abundance_maps = artefacts["nmf"]["abundance_maps"]
score_maps = artefacts["pca"]["score_maps"]
shape = abundance_maps.shape[:2]
n_clusters = abundance_maps.shape[-1]

X_nmf = cluster.nmf_feature_space(abundance_maps, fg_idx)
X_pca = cluster.pca_feature_space(score_maps, fg_idx, n_components=3)
print(f"{len(fg_idx):,} foreground pixels, {n_clusters} segments")

48,294 foreground pixels, 3 segments


## The four methods

The dominant-endmember partition is the primary result; the others are
independent checks on it.

In [3]:
labels = {}
labels["dominant"] = cluster.dominant_endmember_labels(X_nmf)
labels["kmeans_pca"], _ = cluster.run_kmeans(X_pca, n_clusters)
gmm = cluster.run_gmm(X_pca, n_clusters)
labels["gmm"] = gmm["labels"]

if "umap" in artefacts:
    size = cluster.resolve_min_cluster_size(
        config.get("segmentation.hdbscan.min_cluster_size"), len(fg_idx)
    )
    hdb = cluster.run_hdbscan(artefacts["umap"]["embedding"], size)
    labels["hdbscan"] = hdb["labels"]
    print(f"HDBSCAN: {hdb['n_clusters']} clusters, {hdb['noise_pct']:.1f}% unassigned")

for name, values in labels.items():
    print(f"  {name:12s} {len(set(values))} labels")

HDBSCAN: 5 clusters, 55.9% unassigned
  dominant     3 labels
  kmeans_pca   3 labels
  gmm          3 labels
  hdbscan      6 labels


## How much do they agree?

Low agreement is a finding, not a failure: it says the chemistry is not
cleanly separable in every feature space.

In [4]:
restrict = labels["hdbscan"] != -1 if "hdbscan" in labels else None
agreement = cluster.agreement_metrics(labels, restrict_mask=restrict)
agreement

,method_a,method_b,ARI,NMI
0,dominant,kmeans_pca,0.1593,0.1850
1,dominant,gmm,0.4962,0.3716
2,dominant,hdbscan,0.1934,0.2678
3,kmeans_pca,gmm,0.0848,0.1489
4,kmeans_pca,hdbscan,-0.0093,0.0727
5,gmm,hdbscan,0.3888,0.3887


In [5]:
purity = cluster.segment_purity(labels["dominant"], labels["kmeans_pca"])
print(f"pixel-weighted purity : {purity['weighted_purity']:.3f}")
print(f"majority baseline     : {purity['majority_baseline']:.3f}")
purity["per_cluster"]

pixel-weighted purity : 0.693
majority baseline     : 0.667


,cluster,size,majority_segment,purity
0,0,46811,0,0.6869
1,1,1061,1,0.9972
2,2,422,2,0.6137


## Figures and saved labels

In [6]:
from src.viz import segmentation_plots
from src.viz.style import segment_names

label_images = {
    name: cluster.labels_to_image(values, fg_idx, shape)
    for name, values in labels.items()
}
entropy_image = cluster.values_to_image(gmm["entropy"], fg_idx, shape)

segmentation_plots.plot_segmentation_comparison(
    label_images, config.figure_path("03_segmentation_comparison.png"),
    entropy_image=entropy_image,
)
segmentation_plots.plot_segment_purity(
    purity, config.figure_path("03_segment_purity.png")
)
print("figures written")

figures written


Running the stage from the CLI also writes `fossilfly_segments.npz` and the
metrics table that stage 04 reads:

```bash
python scripts/run_pipeline.py --stage segment
```